# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library. You'll use Croissant schema `@id` references to interact with all dataset elements for reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access metadata object and display basic info
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Dataset published on: {metadata.datePublished}")
print(f"Dataset license: {metadata.license}")
print(f"Keywords: {', '.join(metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s. Each Croissant entity is referenced by its `@id`:

In [ ]:
# List all record sets with @id and name
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in metadata.")
else:
    print("Available Record Sets:")
    for rs in record_sets:
        print(f"@id: {rs['@id']}, name: {rs.get('name', 'No name')}")
    
    # Overview fields and columns within each record set
    for rs in record_sets:
        print(f"\nRecordSet @id: {rs['@id']} Columns:")
        columns = rs.get('column', [])
        for col in columns:
            if isinstance(col, dict):
                print(f"  Column @id: {col['@id']}, name: {col.get('name', 'No name')}, dataType: {col.get('dataType', '')}")
            else:
                print(f"  Column @id: {col}")

## 3. Data Extraction
Load data from each record set into Pandas DataFrames for analysis, using their `@id` references.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets if '@id' in rs]
dataframes = {}

for rs_id in record_set_ids:
    # Load records as list of dicts for each record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nRecordSet @id: {rs_id} loaded with columns:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filter numeric values, normalize, and group. Reference all fields and columns via their `@id`s.

In [ ]:
# Example EDA for the first record set with a numeric column
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    print(f"Using RecordSet @id: {first_rs_id}\n")

    # Attempt to find numeric columns by their @id
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        threshold = df[numeric_field_id].median() if not df[numeric_field_id].empty else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Attempt grouping by another field if possible
        group_field_candidates = [col for col in df.columns if col != numeric_field_id]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by {group_field}:")
                print(grouped_df.head())
    else:
        print("No numeric columns found for EDA in this record set.")
else:
    print("No dataframes loaded for EDA.")

## 5. Visualization
Visualize distributions or relationships in the dataset using field `@id`s. Below is an example using matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution for the first record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    df = dataframes[first_rs_id]
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_cols:
        numeric_field_id = numeric_cols[0]
        plt.figure(figsize=(7, 4))
        sns.histplot(df[numeric_field_id], kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Frequency')
        plt.show()
    else:
        print("No numeric column found to visualize.")
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset using `mlcroissant`, explored its structure via Croissant `@id` references, extracted tabular data for each record set, and applied basic exploratory and visualization steps. All fields and columns were referenced using their `@id`s for transparency. For further research or reproducibility, use these IDs explicitly across your workflows.

The dataset is recommended primarily for academic research, genotype-phenotype case study, and clinical correlation exploration, not for ML model training due to limited sample size and scope.